# Model experiments with MLflow

This notebook runs every recommender model with every data processor, logs the experiments to MLflow, and saves the processed datasets and model artifacts.

It is designed for repeatable experiment tracking and DVC-friendly dataset versioning.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split
import yaml

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from recommender.data import DataProcessorContext, RecommenderDataset, load_events
from recommender.models import ModelFactory
from recommender.mlflow_toolkit import MLflowToolkit
from recommender.training import Trainer, hit_rate_at_k, ndcg_at_k

In [ ]:
CONFIG_PATH = ROOT / "configs" / "model.yaml"
MLFLOW_CONFIG_PATH = ROOT / "configs" / "mlflow.yaml"
PROCESSED_DIR = ROOT / "data" / "processed" / "mlflow_experiments"
ARTIFACT_DIR = ROOT / "models" / "mlflow_experiments"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    base_cfg = yaml.safe_load(f)["model"]

with open(MLFLOW_CONFIG_PATH) as f:
    mlflow_cfg = yaml.safe_load(f)["mlflow"]

processor_names = ["weighted", "binary", "implicit"]
model_types = ["ncf", "gmf", "matrix_factorization"]

processor_kwargs_map = {
    "weighted": {},
    "binary": {},
    "implicit": {},
}

mlflow_toolkit = MLflowToolkit(
    tracking_uri=mlflow_cfg.get("tracking_uri"),
    experiment_name=mlflow_cfg.get("experiment_name"),
)
mlflow_toolkit.setup()

base_cfg

In [ ]:
def build_model_hyperparams(cfg: dict) -> dict:
    hyper = cfg.get("hyperparams", {}) or {}
    keys = ("embedding_dim", "hidden_layers", "dropout", "projection_dim", "global_bias")
    return {k: v for k, v in hyper.items() if k in keys}


def save_processed_dataset(df: pd.DataFrame, model_type: str, processor_name: str) -> Path:
    out_path = PROCESSED_DIR / f"{model_type}_{processor_name}_interactions.csv"
    df.to_csv(out_path, index=False)
    try:
        subprocess.run(["dvc", "add", str(out_path)], check=True, capture_output=True, text=True)
    except FileNotFoundError:
        print(f"DVC not available, skipped: dvc add {out_path}")
    except subprocess.CalledProcessError as exc:
        print(f"DVC add failed for {out_path}: {exc.stderr or exc.stdout}")
    return out_path


def train_one_experiment(events: pd.DataFrame, model_type: str, processor_name: str, seed: int) -> dict:
    processor = DataProcessorContext(processor_name, **processor_kwargs_map[processor_name])
    interactions, user2idx, item2idx = processor.process(events, min_interactions=base_cfg.get("min_interactions", 1))

    processed_path = save_processed_dataset(interactions, model_type, processor_name)
    mlflow_toolkit.log_dataset(interactions, name=f"{model_type}_{processor_name}_interactions", source=str(processed_path), context="training")

    num_users = len(user2idx)
    num_items = len(item2idx)
    dataset = RecommenderDataset(interactions, num_items, num_negatives=base_cfg["num_negatives"])
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(seed),
    )

    train_loader = DataLoader(train_dataset, batch_size=base_cfg["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=base_cfg["batch_size"])
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = ModelFactory.create(
        model_type,
        num_users=num_users,
        num_items=num_items,
        **build_model_hyperparams(base_cfg),
    )
    trainer = Trainer(model, base_cfg, device=device)

    last_metrics = {}
    last_loss = 0.0
    for epoch in range(base_cfg["epochs"]):
        last_loss = trainer.train_epoch(train_loader)
        last_metrics = trainer.evaluate(val_loader)

    val_indices = val_dataset.indices
    val_samples = np.array([dataset.samples[i] for i in val_indices[: min(10000, len(val_indices))]])
    positive_only = val_samples[val_samples[:, 2] == 1.0][:, :2].astype(np.int64)
    hr = hit_rate_at_k(model, positive_only[:1000], num_items, k=10, device=device)
    ndcg = ndcg_at_k(model, positive_only[:1000], num_items, k=10, device=device)

    checkpoint_path = ARTIFACT_DIR / f"{model_type}_{processor_name}.pt"
    torch.save(
        {
            "model_type": model_type,
            "processor": processor_name,
            "model_state_dict": model.state_dict(),
            "user2idx": user2idx,
            "item2idx": item2idx,
            "config": base_cfg,
            "metrics": {
                "loss": last_loss,
                "auc_roc": last_metrics["auc_roc"],
                "avg_precision": last_metrics["avg_precision"],
                "hit_rate_at_10": hr,
                "ndcg_at_10": ndcg,
            },
        },
        checkpoint_path,
    )

    mlflow_toolkit.log_artifact(checkpoint_path)
    mlflow_toolkit.log_metrics(
        {
            "train_loss": float(last_loss),
            "auc_roc": float(last_metrics["auc_roc"]),
            "avg_precision": float(last_metrics["avg_precision"]),
            "hit_rate_at_10": float(hr),
            "ndcg_at_10": float(ndcg),
        }
    )

    return {
        "model_type": model_type,
        "processor": processor_name,
        "artifact": str(checkpoint_path),
        "processed_data": str(processed_path),
        "train_loss": float(last_loss),
        "auc_roc": float(last_metrics["auc_roc"]),
        "avg_precision": float(last_metrics["avg_precision"]),
        "hit_rate_at_10": float(hr),
        "ndcg_at_10": float(ndcg),
    }


In [ ]:
raw_events_path = ROOT / base_cfg.get("raw_events_path", "data/raw/events.csv")
events = load_events(str(raw_events_path))
results = []

for model_type in model_types:
    for processor_name in processor_names:
        run_name = f"{model_type}-{processor_name}"
        with mlflow_toolkit.start_run(run_name=run_name, tags={"model_type": model_type, "processor": processor_name}):
            mlflow_toolkit.log_params({
                "model_type": model_type,
                "processor": processor_name,
                "seed": base_cfg["seed"],
                "batch_size": base_cfg["batch_size"],
                "learning_rate": base_cfg["learning_rate"],
                "epochs": base_cfg["epochs"],
                "num_negatives": base_cfg["num_negatives"],
                "min_interactions": base_cfg.get("min_interactions", 1),
            })
            result = train_one_experiment(events, model_type, processor_name, seed=base_cfg["seed"])
            results.append(result)

results_df = pd.DataFrame(results)
results_df.sort_values(["model_type", "processor"]).reset_index(drop=True)

## What gets saved

For each `(model, processor)` combination this notebook produces:

- a processed dataframe saved under `data/processed/mlflow_experiments/`
- a DVC-tracked file via `dvc add` when available
- a PyTorch checkpoint under `models/mlflow_experiments/`
- MLflow params, metrics, dataset input, and artifact logs

This gives us a simple comparison matrix across all models and all preprocessing strategies.